# faultmap - Find Where Your LLM Is Failing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabonavarroo/faultmap/blob/main/notebooks/tutorial.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/faultmap.svg)](https://pypi.org/project/faultmap/)

**faultmap** helps you answer three practical questions:
- Where is my model failing?
- What is my test suite missing?
- Which model wins on each semantic slice?

This notebook is optimized for GitHub readers who want to preview the library's main usage patterns.
Open it in Colab if you want to run the mock path end-to-end.

What it covers:
- **Mode 1**: Bring your own scores
- **Mode 2**: Reference-based scoring
- **Mode 3**: Reference-free scoring
- **Coverage auditing**
- **Model comparison**
    


![faultmap 3D embedding visualization](https://raw.githubusercontent.com/gabonavarroo/faultmap/main/docs/assets/faultmap_demo.gif)

> The animation above comes from the 3D embedding visualization demo.
> **[Open the interactive visualization notebook ->](https://github.com/gabonavarroo/faultmap/blob/main/notebooks/visualization_demo.ipynb)**
    


In [ ]:
# Optional: run this first in Colab.
# If you are only previewing the notebook on GitHub, you can skip it.
%pip install -q faultmap[rich]
    


## Tiny Editable Example

Before the larger demos below, here is the smallest useful input shape.
Replace these lists with your own made-up prompts, responses, scores, or references when you want a quick smoke test.
    


In [ ]:
# Minimal data shape for your own quick tests.
# Replace these lists when you want to try the library with your own examples.

EXAMPLE_PROMPTS = [
    "What are the GDPR breach reporting requirements?",
    "How do I reset a team member's password?",
]
EXAMPLE_RESPONSES = [
    "Notify the relevant authority within 72 hours when required.",
    "Open Settings > Security and click Reset password.",
]
EXAMPLE_SCORES = [0.25, 0.95]
EXAMPLE_REFERENCES = [
    "GDPR can require notifying the supervisory authority within 72 hours.",
    "To reset a password, open Settings > Security and start the reset flow.",
]

print("Edit EXAMPLE_PROMPTS / RESPONSES / SCORES / REFERENCES to match your own data.")
    


## Default Path: Mock Everything

The cells below use a deterministic mock embedder and a mocked LLM client.
That means the notebook can showcase the real `SliceAnalyzer` pipeline without any API key.

The mock setup is aligned with the current internals:
- `faultmap.analyzer.get_embedder`
- `faultmap.analyzer.AsyncLLMClient`

Each section sets its clustering method explicitly so the synthetic examples stay readable.
The slice-analysis demos use `agglomerative`, while coverage and comparison switch to `hdbscan` where it produces cleaner topic-level outputs.


In [ ]:
import hashlib
import logging
import warnings
from unittest.mock import AsyncMock, patch

import numpy as np

from faultmap import SliceAnalyzer
from faultmap.embeddings import Embedder

warnings.filterwarnings(
    "ignore",
    message=r"The default value of `copy` will change from False to True in 1\.10\.",
    category=FutureWarning,
)

logging.getLogger("faultmap.coverage.detector").setLevel(logging.ERROR)

LEGAL_KEYWORDS = (
    "gdpr",
    "hipaa",
    "soc2",
    "soc 2",
    "pci",
    "ccpa",
    "iso 27001",
    "compliance",
    "regulation",
    "audit",
    "breach",
    "consent",
    "data protection",
)
ACCOUNT_KEYWORDS = (
    "password",
    "2fa",
    "sso",
    "saml",
    "account",
    "security",
    "email",
    "phone",
    "settings",
    "webhook",
    "ldap",
)
BILLING_KEYWORDS = (
    "pricing",
    "plan",
    "tier",
    "subscription",
    "invoice",
    "refund",
    "billing",
    "charge",
    "payment",
)
INTEGRATION_KEYWORDS = (
    "integrate",
    "integration",
    "github actions",
    "langchain",
    "pytest",
    "deepeval",
    "weights & biases",
    "mlflow",
    "bedrock",
    "azure openai",
    "hugging face",
    "ollama",
)

DIVERSE_LEGAL_ANSWERS = [
    "Start with a data inventory, breach workflow, and regulator-specific retention rules.",
    "The answer depends on consent handling, data subject rights, and audit readiness.",
    "Focus on vendor contracts, lawful basis, and documented access controls.",
    "Expect different requirements for incident reporting, encryption, and review cadence.",
]
CONSISTENT_SETUP_ANSWER = "Open Settings > Security and follow the setup wizard for your team."


def infer_topic(text: str) -> str:
    lower = text.lower()
    if any(keyword in lower for keyword in LEGAL_KEYWORDS):
        return "legal"
    if any(keyword in lower for keyword in ACCOUNT_KEYWORDS):
        return "account"
    if any(keyword in lower for keyword in BILLING_KEYWORDS):
        return "billing"
    if any(keyword in lower for keyword in INTEGRATION_KEYWORDS):
        return "integration"
    return "general"


class TutorialMockEmbedder(Embedder):
    """Deterministic embedder with topic-aware clusters for tutorial data."""

    DIM = 64

    def __init__(self) -> None:
        seed_rng = np.random.default_rng(7)
        self._centers = {}
        for topic in ("legal", "account", "billing", "integration", "general"):
            center = seed_rng.standard_normal(self.DIM)
            self._centers[topic] = center / (np.linalg.norm(center) + 1e-10)

    def _noise(self, text: str, usage: str) -> np.ndarray:
        digest = hashlib.md5(f"{usage}::{text}".encode("utf-8")).digest()
        seed = int.from_bytes(digest[:8], "little", signed=False)
        rng = np.random.default_rng(seed)
        return rng.standard_normal(self.DIM)

    def embed(self, texts: list[str], *, usage: str = "generic") -> np.ndarray:
        if not texts:
            return np.empty((0, self.DIM), dtype=np.float32)

        usage_noise = 0.02 if usage == "query" else 0.28
        rows = []
        for text in texts:
            topic = infer_topic(text)
            center = self._centers[topic] * 3.0
            noise = self._noise(text, usage) * usage_noise
            vec = center + noise
            vec = vec / (np.linalg.norm(vec) + 1e-10)
            rows.append(vec.astype(np.float32))
        return np.vstack(rows)

    @property
    def dimension(self) -> int:
        return self.DIM


def label_for_topic(topic: str) -> str:
    if topic == "legal":
        return (
            "Name: Legal Compliance Questions\n"
            "Description: Prompts asking about regulations, audits, and compliance requirements.\n"
            "Root Cause: The model lacks policy-specific detail and over-generalizes legal guidance.\n"
            "Suggested Fix: Add compliance-specific instructions and require policy-grounded answers before responding."
        )
    if topic == "account":
        return (
            "Name: Account Recovery And Setup\n"
            "Description: Prompts about authentication, account recovery, and workspace setup.\n"
            "Root Cause: The model needs clearer procedural steps for account and security workflows.\n"
            "Suggested Fix: Add concise setup playbooks and explicit fallback steps for security-related requests."
        )
    if topic == "billing":
        return (
            "Name: Pricing And Billing Questions\n"
            "Description: Prompts about plans, subscriptions, invoices, and billing disputes.\n"
            "Root Cause: The model mixes plan details with policy answers and misses billing-specific nuance.\n"
            "Suggested Fix: Add current plan definitions, refund policy guidance, and invoice troubleshooting steps."
        )
    if topic == "integration":
        return (
            "Name: Integration Coverage Gap\n"
            "Description: Prompts about integrations, tooling, and provider compatibility that are absent from the test set.\n"
            "Root Cause: The current eval set does not include enough integration-style prompts.\n"
            "Suggested Fix: Add representative prompts for framework integrations and provider-specific setup questions."
        )
    return (
        "Name: General Questions\n"
        "Description: A broad cluster of prompts that do not match a more specific tutorial topic.\n"
        "Root Cause: The model is relying on generic heuristics instead of a domain-specific policy.\n"
        "Suggested Fix: Add a short domain framing instruction before answering generic requests."
    )


async def mock_label_completion(messages, **kwargs) -> str:
    user_text = "\n".join(message["content"] for message in messages if message["role"] == "user")
    return label_for_topic(infer_topic(user_text))


def make_mode3_samples(prompts: list[str], n_samples: int) -> list[str]:
    samples = []
    for prompt in prompts:
        if infer_topic(prompt) == "legal":
            for idx in range(n_samples):
                samples.append(DIVERSE_LEGAL_ANSWERS[idx % len(DIVERSE_LEGAL_ANSWERS)])
        else:
            for _ in range(n_samples):
                samples.append(CONSISTENT_SETUP_ANSWER)
    return samples


def build_mock_analyzer(*, sampled_responses=None, **kwargs):
    mock_embedder = TutorialMockEmbedder()
    mock_client = AsyncMock()
    mock_client.complete = AsyncMock(side_effect=mock_label_completion)
    mock_client.complete_batch = AsyncMock(return_value=sampled_responses or [])

    with patch("faultmap.analyzer.get_embedder", return_value=mock_embedder), patch(
        "faultmap.analyzer.AsyncLLMClient", return_value=mock_client
    ):
        analyzer = SliceAnalyzer(**kwargs)

    return analyzer, mock_client


print("Mock helpers ready. The next sections run without any API key.")
    


## Mode 1: Bring Your Own Scores

Use this when you already have scores from DeepEval, Ragas, human review, or any LLM-as-judge pipeline.
The library uses those scores to find semantic slices with a significantly higher failure rate.
    


In [ ]:
legal_topics = ["GDPR", "HIPAA", "SOC2", "PCI-DSS", "CCPA", "ISO 27001"]
account_topics = ["password", "2FA", "email", "phone", "settings", "account"]
pricing_topics = ["Basic", "Pro", "Enterprise", "Team", "Starter", "Business"]

legal_templates = [
    "What are the legal requirements for {topic} compliance?",
    "Give me a checklist for {topic} controls.",
    "Which audits are required for {topic}?",
    "How should a startup prepare for a {topic} review?",
    "What evidence do I need for {topic} certification?",
]
account_templates = [
    "How do I reset my {topic}?",
    "Walk me through fixing a broken {topic} login.",
    "What steps should support follow for {topic} recovery?",
    "How do I troubleshoot a {topic} issue for a teammate?",
    "What is the fastest way to update my {topic} settings?",
]
pricing_templates = [
    "What is included in the {topic} plan?",
    "How does the {topic} tier differ from the others?",
    "Which users should upgrade to the {topic} package?",
    "What billing limits come with the {topic} subscription?",
    "Summarize the {topic} plan for a buyer.",
]

mode1_prompts = (
    [template.format(topic=topic) for topic in legal_topics for template in legal_templates]
    + [template.format(topic=topic) for topic in account_topics for template in account_templates]
    + [template.format(topic=topic) for topic in pricing_topics for template in pricing_templates]
)
mode1_responses = [f"Draft answer for: {prompt}" for prompt in mode1_prompts]
mode1_scores = [0.18] * (len(legal_topics) * len(legal_templates))
mode1_scores += [0.90] * (len(account_topics) * len(account_templates))
mode1_scores += [0.93] * (len(pricing_topics) * len(pricing_templates))

analyzer1, _ = build_mock_analyzer(
    model="gpt-4o-mini",
    min_slice_size=5,
    failure_threshold=0.5,
    clustering_method="agglomerative",
)
report1 = analyzer1.analyze(mode1_prompts, mode1_responses, scores=mode1_scores)

print(report1)
for slice_ in report1.slices:
    print(f"- {slice_.name}: {slice_.failure_rate:.0%} failure rate across {slice_.size} prompts")


## Inspecting Results

`AnalysisReport` gives you the slice metadata, representative prompts, root-cause summary, and suggested remediation.
    


In [ ]:
print(f"Total prompts: {report1.total_prompts}")
print(f"Total failures: {report1.total_failures}")
print(f"Baseline failure rate: {report1.baseline_failure_rate:.1%}")
print(f"Significant slices: {report1.num_significant}")
print()

if report1.slices:
    first_slice = report1.slices[0]
    print(f"Top slice: {first_slice.name}")
    print(f"Description: {first_slice.description}")
    print(f"Root cause: {first_slice.root_cause}")
    print(f"Suggested fix: {first_slice.suggested_remediation}")
    print(f"Representative prompts: {first_slice.representative_prompts[:3]}")
    print(f"JSON keys: {list(report1.to_dict().keys())}")
    


## Mode 2: Reference-Based Scoring

Provide ground-truth answers and let `faultmap` score responses by semantic similarity.
This is useful for QA, RAG, support answer quality, and translation-style tasks.
    


In [ ]:
mode2_topics = ["GDPR", "HIPAA", "CCPA", "SOC2", "PCI-DSS"]
mode2_features = ["SSO", "2FA", "webhooks", "API keys", "SAML"]

legal_templates = [
    "What does {topic} regulate?",
    "Summarize the main obligations under {topic}.",
    "What controls should a team document for {topic}?",
    "Which data handling rules matter most for {topic}?",
    "What evidence would an auditor ask for under {topic}?",
    "How would you explain {topic} to a support lead?",
]
setup_templates = [
    "How do I set up {feature}?",
    "Where should I start when enabling {feature}?",
    "What steps does an admin follow to configure {feature}?",
    "How do I roll out {feature} to the whole workspace?",
    "What troubleshooting steps help when {feature} setup fails?",
    "How would support explain {feature} activation?",
]

mode2_prompts = (
    [template.format(topic=topic) for topic in mode2_topics for template in legal_templates]
    + [template.format(feature=feature) for feature in mode2_features for template in setup_templates]
)
mode2_references = (
    [f"{topic} defines data handling, audit, or governance requirements for that compliance area." for topic in mode2_topics for _ in legal_templates]
    + [f"To set up {feature}, open Settings and follow the guided configuration flow." for feature in mode2_features for _ in setup_templates]
)
mode2_responses = (
    ["Please check the pricing page for the latest plans and discounts." for _ in range(len(mode2_topics) * len(legal_templates))]
    + [f"To set up {feature}, open Settings and follow the guided configuration flow." for feature in mode2_features for _ in setup_templates]
)

analyzer2, _ = build_mock_analyzer(
    model="gpt-4o-mini",
    min_slice_size=5,
    failure_threshold=0.5,
    clustering_method="agglomerative",
)
report2 = analyzer2.analyze(mode2_prompts, mode2_responses, references=mode2_references)

print(report2)
for slice_ in report2.slices:
    print(f"- {slice_.name}: adjusted p-value = {slice_.adjusted_p_value:.4f}")


## Mode 3: Reference-Free Scoring

When you do not have labels or references, `faultmap` can score reliability with semantic entropy and self-consistency.
The mock below makes legal prompts diverse and uncertain, while setup prompts stay consistent.
    


In [ ]:
mode3_topics = ["GDPR", "HIPAA", "CCPA", "SOC2"]
mode3_features = ["SSO", "2FA", "webhooks", "LDAP"]

legal_templates = [
    "Explain {topic} compliance requirements in detail.",
    "What can go wrong if a team misunderstands {topic}?",
    "How should a startup prepare for a {topic} audit?",
    "What policy gaps usually appear during a {topic} review?",
    "How would you brief leadership on {topic} risk?",
]
setup_templates = [
    "How do I configure {feature} for my team?",
    "What rollout steps help when enabling {feature}?",
    "How should IT document a {feature} setup?",
    "Which admin checks matter before launching {feature}?",
    "How would support explain {feature} onboarding?",
]

mode3_prompts = (
    [template.format(topic=topic) for topic in mode3_topics for template in legal_templates]
    + [template.format(feature=feature) for feature in mode3_features for template in setup_templates]
)
mode3_responses = [
    DIVERSE_LEGAL_ANSWERS[0] if infer_topic(prompt) == "legal" else CONSISTENT_SETUP_ANSWER
    for prompt in mode3_prompts
]
mode3_samples = make_mode3_samples(mode3_prompts, n_samples=4)

analyzer3, _ = build_mock_analyzer(
    model="gpt-4o-mini",
    min_slice_size=5,
    n_samples=4,
    temperature=1.0,
    clustering_method="agglomerative",
    sampled_responses=mode3_samples,
)
report3 = analyzer3.analyze(mode3_prompts, mode3_responses)

print(report3)
for slice_ in report3.slices:
    print(f"- {slice_.name}: {slice_.failure_rate:.0%} failure rate")


## Coverage Auditing

Compare your test prompts with production prompts to find semantic gaps your current eval set never touches.
    


In [ ]:
account_actions = ["upgrade", "downgrade", "cancel", "pause", "reactivate"]
plan_names = ["Basic", "Pro", "Enterprise", "Team"]
integration_tools = ["GitHub Actions", "LangChain", "Pytest", "DeepEval", "Weights & Biases", "MLflow"]
providers = ["AWS Bedrock", "Azure OpenAI", "Hugging Face", "Ollama"]

account_templates = [
    "How do I {action} my account?",
    "What steps should support follow to {action} a workspace subscription?",
    "How would you explain how to {action} an account to a teammate?",
    "What admin workflow is used to {action} billing access?",
]
plan_templates = [
    "What does the {plan} plan include?",
    "Who should buy the {plan} tier?",
    "How does the {plan} plan affect billing?",
    "What support limits come with the {plan} package?",
    "Summarize the {plan} plan for procurement.",
]
integration_templates = [
    "How do I integrate faultmap with {tool}?",
    "What setup steps are needed for a {tool} integration?",
    "How would a team productionize faultmap with {tool}?",
]
provider_templates = [
    "Can I use faultmap with {provider}?",
    "What should I configure before running faultmap on {provider}?",
    "How would you explain provider support for {provider}?",
]

test_prompts = (
    [template.format(action=action) for action in account_actions for template in account_templates]
    + [template.format(plan=plan) for plan in plan_names for template in plan_templates]
)
production_prompts = (
    [
        template.format(action=action)
        for action in ["upgrade", "downgrade", "cancel", "pause", "modify"]
        for template in account_templates
    ]
    + [template.format(plan=plan) for plan in plan_names for template in plan_templates]
    + [template.format(tool=tool) for tool in integration_tools for template in integration_templates]
    + [template.format(provider=provider) for provider in providers for template in provider_templates]
)

coverage_analyzer, _ = build_mock_analyzer(
    model="gpt-4o-mini",
    min_slice_size=3,
    clustering_method="hdbscan",
)
coverage = coverage_analyzer.audit_coverage(
    test_prompts=test_prompts,
    production_prompts=production_prompts,
    distance_threshold=0.80,
    min_gap_size=5,
)

print(coverage)
for gap in coverage.gaps:
    print(f"- {gap.name}: {gap.size} uncovered prompts")


## Model Comparison

Use `compare_models()` when you want to see which model wins overall and which model wins inside each slice.
    


In [ ]:
billing_topics = [
    "double charge",
    "failed payment",
    "refund request",
    "invoice error",
    "subscription cancel",
    "billing dispute",
]

legal_templates = [
    "What are the compliance requirements for {topic}?",
    "How would you summarize the main audit steps for {topic}?",
    "Which evidence matters most for a {topic} review?",
    "What policy gaps usually break {topic} readiness?",
    "How should support answer a {topic} question from a customer?",
]
billing_templates = [
    "My {topic} needs to be resolved urgently.",
    "What workflow should finance follow for a {topic}?",
    "How would support triage a {topic}?",
    "What does a good response look like for a {topic}?",
    "What escalation path helps with a {topic}?",
]
pricing_templates = [
    "What is included in the {topic} plan?",
    "How should I explain the {topic} tier to finance?",
    "What limits come with the {topic} subscription?",
    "Which users are a fit for the {topic} plan?",
    "How does the {topic} package affect support access?",
]

comparison_prompts = (
    [template.format(topic=topic) for topic in legal_topics for template in legal_templates]
    + [template.format(topic=topic) for topic in billing_topics for template in billing_templates]
    + [template.format(topic=topic) for topic in pricing_topics for template in pricing_templates]
)
comparison_responses_a = [f"[GPT-4o] {prompt}" for prompt in comparison_prompts]
comparison_responses_b = [f"[GPT-4o-mini] {prompt}" for prompt in comparison_prompts]
comparison_scores_a = [0.85] * (len(legal_topics) * len(legal_templates))
comparison_scores_a += [0.30] * (len(billing_topics) * len(billing_templates))
comparison_scores_a += [0.82] * (len(pricing_topics) * len(pricing_templates))
comparison_scores_b = [0.25] * (len(legal_topics) * len(legal_templates))
comparison_scores_b += [0.90] * (len(billing_topics) * len(billing_templates))
comparison_scores_b += [0.82] * (len(pricing_topics) * len(pricing_templates))

comparison_analyzer, _ = build_mock_analyzer(
    model="gpt-4o-mini",
    min_slice_size=5,
    failure_threshold=0.5,
    clustering_method="hdbscan",
)
comparison = comparison_analyzer.compare_models(
    prompts=comparison_prompts,
    responses_a=comparison_responses_a,
    responses_b=comparison_responses_b,
    scores_a=comparison_scores_a,
    scores_b=comparison_scores_b,
    model_a_name="GPT-4o",
    model_b_name="GPT-4o-mini",
)

print(comparison)
print(f"Global winner: {comparison.global_winner}")
print(f"Global advantage rate: {comparison.global_advantage_rate:.0%}")
for slice_ in comparison.slices:
    print(f"- {slice_.name}: winner = {slice_.winner}, adj. p = {slice_.adjusted_p_value:.4f}")


## Bring Your Own Key

The cells below are explicit templates for real usage.
Replace the `YOUR_*` values, rerun the setup cell, and then run the mode you care about.
They intentionally print guidance instead of making accidental API calls until you replace the placeholder key.
    


In [ ]:
import os

from faultmap import SliceAnalyzer

YOUR_API_KEY = "ADD_YOUR_OPENAI_API_KEY"
REAL_MODEL = "gpt-4o-mini"
REAL_EMBEDDING_MODEL = "text-embedding-3-small"  # or "all-MiniLM-L6-v2" if you want local embeddings

if YOUR_API_KEY.startswith("ADD_YOUR"):
    print("Replace YOUR_API_KEY, then rerun this cell to create `real_analyzer`.")
else:
    os.environ["OPENAI_API_KEY"] = YOUR_API_KEY
    real_analyzer = SliceAnalyzer(
        model=REAL_MODEL,
        embedding_model=REAL_EMBEDDING_MODEL,
        significance_level=0.05,
        min_slice_size=10,
        failure_threshold=0.5,
    )
    print("`real_analyzer` is ready.")
    


In [ ]:
# Mode 1 template: replace these example values with your own eval outputs.
YOUR_PROMPTS = [
    "What are the GDPR reporting requirements for a breach?",
    "How do I reset a workspace password?",
]
YOUR_RESPONSES = [
    "Report the breach within 72 hours when required.",
    "Open Settings > Security and start the reset flow.",
]
YOUR_SCORES = [0.20, 0.95]

if "real_analyzer" not in globals():
    print("Run the setup cell above after replacing YOUR_API_KEY.")
else:
    real_report_mode1 = real_analyzer.analyze(
        prompts=YOUR_PROMPTS,
        responses=YOUR_RESPONSES,
        scores=YOUR_SCORES,
    )
    print(real_report_mode1.summary())
    


In [ ]:
# Mode 2 template: replace these example values with your own prompts, responses, and references.
YOUR_PROMPTS = [
    "What does HIPAA regulate?",
    "How do I configure SSO for my workspace?",
]
YOUR_RESPONSES = [
    "HIPAA is a healthcare privacy and data handling regulation.",
    "Open Settings and follow the SSO configuration steps.",
]
YOUR_REFERENCES = [
    "HIPAA governs protected health information and related compliance requirements.",
    "To configure SSO, open Settings and complete the SSO setup flow.",
]

if "real_analyzer" not in globals():
    print("Run the setup cell above after replacing YOUR_API_KEY.")
else:
    real_report_mode2 = real_analyzer.analyze(
        prompts=YOUR_PROMPTS,
        responses=YOUR_RESPONSES,
        references=YOUR_REFERENCES,
    )
    print(real_report_mode2.summary())
    


In [ ]:
# Mode 3 template: replace these example values with your own prompts and model responses.
YOUR_PROMPTS = [
    "Explain SOC2 audit requirements for a startup.",
    "How do I set up 2FA for my team?",
]
YOUR_RESPONSES = [
    "SOC2 involves controls, audits, and evidence collection across the system.",
    "Open Settings > Security and follow the 2FA setup wizard.",
]

if "real_analyzer" not in globals():
    print("Run the setup cell above after replacing YOUR_API_KEY.")
else:
    real_report_mode3 = real_analyzer.analyze(
        prompts=YOUR_PROMPTS,
        responses=YOUR_RESPONSES,
    )
    print(real_report_mode3.summary())
    


In [ ]:
# Coverage template: replace these example lists with your own test and production prompts.
YOUR_TEST_PROMPTS = [
    "How do I upgrade my plan?",
    "How do I reset my account password?",
]
YOUR_PRODUCTION_PROMPTS = [
    "How do I upgrade my plan?",
    "Can I integrate faultmap with GitHub Actions?",
    "How do I reset my account password?",
]

if "real_analyzer" not in globals():
    print("Run the setup cell above after replacing YOUR_API_KEY.")
else:
    real_coverage = real_analyzer.audit_coverage(
        test_prompts=YOUR_TEST_PROMPTS,
        production_prompts=YOUR_PRODUCTION_PROMPTS,
    )
    print(real_coverage.summary())
    


In [ ]:
# Comparison template: replace these example values with two models scored on the same prompts.
YOUR_PROMPTS = [
    "What are the GDPR breach reporting requirements?",
    "How do I resolve a billing dispute?",
]
YOUR_RESPONSES_A = [
    "Model A answer about GDPR reporting timelines.",
    "Model A answer about billing escalation.",
]
YOUR_RESPONSES_B = [
    "Model B answer about GDPR reporting timelines.",
    "Model B answer about billing escalation.",
]
YOUR_SCORES_A = [0.90, 0.30]
YOUR_SCORES_B = [0.25, 0.92]
YOUR_MODEL_A_NAME = "GPT-4o"
YOUR_MODEL_B_NAME = "GPT-4o-mini"

if "real_analyzer" not in globals():
    print("Run the setup cell above after replacing YOUR_API_KEY.")
else:
    real_comparison = real_analyzer.compare_models(
        prompts=YOUR_PROMPTS,
        responses_a=YOUR_RESPONSES_A,
        responses_b=YOUR_RESPONSES_B,
        scores_a=YOUR_SCORES_A,
        scores_b=YOUR_SCORES_B,
        model_a_name=YOUR_MODEL_A_NAME,
        model_b_name=YOUR_MODEL_B_NAME,
    )
    print(real_comparison.summary())
    


## Next Steps

- **[README](https://github.com/gabonavarroo/faultmap#readme)** for API details, scoring options, and parameter tuning
- **[examples/](https://github.com/gabonavarroo/faultmap/tree/main/examples)** for script-based usage outside notebooks
- **[issues](https://github.com/gabonavarroo/faultmap/issues)** for bugs, edge cases, and feature requests
    
